# P9 — Visual Question Answering (VQA)
**Same pretrained ViLT checkpoint evaluated on VQA v2 and CLEVR.**

This notebook is prepared for Kaggle. It does **not** contain fabricated results. Add the image/question/annotation datasets through **Add Input**, enable Internet if available (or add pretrained ViLT model files as input), set the paths below, then Run All.

Original model: https://github.com/dandelin/ViLT · checkpoint: https://huggingface.co/dandelin/vilt-b32-finetuned-vqa

In [ ]:
import os, json, csv, random, re
from pathlib import Path
import torch
from PIL import Image
from transformers import ViltProcessor, ViltForQuestionAnswering
print('PyTorch:',torch.__version__)
print('Inputs:')
for p in Path('/kaggle/input').iterdir(): print(' -',p)

## 1. Set dataset paths
Use the right sidebar **Input** to find your actual paths. The VQA v2 *validation* questions and annotations are needed to compute scores. CLEVR validation questions contain answers.

In [ ]:
VQA_IMAGES = '/kaggle/input/REPLACE/val2014'
VQA_QUESTIONS = '/kaggle/input/REPLACE/v2_OpenEnded_mscoco_val2014_questions.json'
VQA_ANNOTATIONS = '/kaggle/input/REPLACE/v2_mscoco_val2014_annotations.json'
CLEVR_IMAGES = '/kaggle/input/REPLACE/CLEVR_v1.0/images/val'
CLEVR_QUESTIONS = '/kaggle/input/REPLACE/CLEVR_v1.0/questions/CLEVR_val_questions.json'
MODEL_PATH = 'dandelin/vilt-b32-finetuned-vqa'  # change to local Kaggle model folder if Internet is unavailable
SAMPLE_SIZE = 30
SEED = 42
OUTPUT = Path('/kaggle/working/vqa_results')
OUTPUT.mkdir(parents=True, exist_ok=True)
for p in [VQA_IMAGES,VQA_QUESTIONS,VQA_ANNOTATIONS,CLEVR_IMAGES,CLEVR_QUESTIONS]:
    if not Path(p).exists(): print('MISSING:',p)

## 2. Load the existing pretrained model
If this fails with a network error, add the **model weights** as Kaggle input and set `MODEL_PATH` to the folder containing `config.json` and the model weights. Adding a notebook as input is not the same as adding model weights.

In [ ]:
processor = ViltProcessor.from_pretrained(MODEL_PATH)
model = ViltForQuestionAnswering.from_pretrained(MODEL_PATH)
model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print('Model ready on',device)

## 3. Read the two validation datasets

In [ ]:
def read_examples(dataset):
    if dataset == 'vqa':
        questions=json.loads(Path(VQA_QUESTIONS).read_text())['questions']
        anns=json.loads(Path(VQA_ANNOTATIONS).read_text())['annotations']
        by_id={int(a['question_id']):a for a in anns}
        data=[]
        for q in questions:
            a=by_id.get(int(q['question_id']))
            if a is None: continue
            image=Path(VQA_IMAGES)/f"COCO_val2014_{int(q['image_id']):012d}.jpg"
            if image.exists(): data.append((image,q['question'],[x['answer'] for x in a['answers']]))
    else:
        questions=json.loads(Path(CLEVR_QUESTIONS).read_text())['questions']
        data=[]
        for q in questions:
            image=Path(CLEVR_IMAGES)/q['image_filename']
            if image.exists() and 'answer' in q: data.append((image,q['question'],[str(q['answer'])]))
    random.Random(SEED).shuffle(data)
    if not data: raise ValueError('No labeled images found for '+dataset+'; check dataset paths.')
    return data[:SAMPLE_SIZE] if SAMPLE_SIZE>0 else data

def norm(x): return ' '.join(str(x).lower().strip().split())
def metric(dataset,pred,answers):
    p=norm(pred)
    if dataset=='clevr': return float(p==norm(answers[0]))
    return sum(min(1.0,sum(norm(a)==p for j,a in enumerate(answers) if j!=i)/3.0) for i in range(len(answers)))/len(answers)
print('VQA samples:',len(read_examples('vqa')))
print('CLEVR samples:',len(read_examples('clevr')))

## 4. Run inference and calculate measured accuracy
VQA uses leave-one-out agreement (with simplified text normalization); CLEVR uses exact match. These metrics are different, so interpret comparisons cautiously.

In [ ]:
summaries=[]
for dataset in ['vqa','clevr']:
    examples=read_examples(dataset)
    rows=[]
    for idx,(path,question,answers) in enumerate(examples,1):
        with Image.open(path) as image:
            batch=processor(images=image.convert('RGB'),text=question,return_tensors='pt',truncation=True)
        batch={k:v.to(device) for k,v in batch.items()}
        with torch.inference_mode(): logits=model(**batch).logits
        pred=model.config.id2label[int(logits.argmax(-1).item())]
        score=metric(dataset,pred,answers)
        rows.append({'image':str(path),'question':question,'prediction':pred,'answers':' | '.join(answers),'score':round(score,4)})
        if idx%10==0: print(dataset,idx,'/',len(examples))
    with (OUTPUT/f'{dataset}_predictions.csv').open('w',newline='',encoding='utf-8') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
    summaries.append({'dataset':dataset,'samples':len(rows),'accuracy':sum(r['score'] for r in rows)/len(rows)})
with (OUTPUT/'summary.csv').open('w',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=['dataset','samples','accuracy']);writer.writeheader();writer.writerows(summaries)
print('ACTUAL MEASURED RESULTS:')
for x in summaries: print(x['dataset'],f"{x['accuracy']:.2%}",f"({x['samples']} samples)")

## 5. Plot and export results
Download `summary.csv`, prediction CSVs, and `accuracy_chart.png` from `/kaggle/working/vqa_results/` and put the actual values into the slides.

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(6,4))
ax.bar([x['dataset'].upper() for x in summaries],[100*x['accuracy'] for x in summaries])
ax.set_ylim(0,100);ax.set_ylabel('Accuracy (%)');ax.set_title('ViLT: Validation subset results')
fig.tight_layout();fig.savefig(OUTPUT/'accuracy_chart.png',dpi=180)
plt.show()
print('Saved to',OUTPUT)